# 02 - Modelado Supervisado
Implementación de múltiples modelos de regresión con Scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Agregar ruta de src al path
sys.path.insert(0, r'C:\Users\Arturo\Desktop\Prueba1\src')

# Importar módulos personalizados
from prueba.data_preprocessing import (
    cargar_datos_crudos, 
    crear_features_empleado, 
    preparar_datos_para_ml
)
from prueba.model_training import (
    entrenar_modelos_regresion, 
    obtener_predicciones,
    obtener_features_importance
)

# Configuración
sns.set_theme(style="whitegrid")
seed = 42
np.random.seed(seed)
project_root = r'C:\Users\Arturo\Desktop\Prueba1'

print("Módulos importados exitosamente")

## 1. Carga y Preparación de Datos

In [ ]:
# Cargar datos crudos
print("Cargando datos...")
datos_crudos = cargar_datos_crudos(os.path.join(project_root, 'data/01_raw/'))

# Crear dataset consolidado
df_features = crear_features_empleado(datos_crudos)
print(f"Dataset consolidado: {df_features.shape}")
print(f"\nCaracterísticas: {list(df_features.columns)}")

# Preparar datos para ML
print("\nPreparando datos para modelos ML...")
datos_ml = preparar_datos_para_ml(df_features, test_size=0.2, random_state=seed)

X_train = datos_ml['X_train']
X_test = datos_ml['X_test']
y_train = datos_ml['y_train']
y_test = datos_ml['y_test']
feature_names = datos_ml['feature_names']

print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")

## 2. Entrenamiento de Modelos

In [ ]:
print("Entrenando múltiples modelos de regresión...\n")
modelos = entrenar_modelos_regresion(X_train, y_train, random_state=seed)

print(f"\nModelos entrenados:")
for nombre in modelos.keys():
    print(f"  ✓ {nombre}")

# Mostrar puntuaciones en training
print("\nPuntuaciones R² en Training:")
for nombre, modelo in modelos.items():
    score = modelo.score(X_train, y_train)
    print(f"  {nombre}: {score:.4f}")

## 3. Predicciones en Test

In [ ]:
# Obtener predicciones
predicciones_train = obtener_predicciones(modelos, X_train)
predicciones_test = obtener_predicciones(modelos, X_test)

print("Predicciones obtenidas para train y test")
print(f"\nModelos con predicciones:")
for nombre in predicciones_test.keys():
    print(f"  ✓ {nombre} - {len(predicciones_test[nombre])} predicciones")

# Comparar primeras 10 predicciones
print("\nMuestras de predicciones (primeras 10 en test):")
df_comparacion = pd.DataFrame({'y_real': y_test.iloc[:10].values})

for nombre, pred in predicciones_test.items():
    df_comparacion[f'{nombre}'] = pred[:10]
    df_comparacion[f'{nombre}_error'] = np.abs(pred[:10] - df_comparacion['y_real'].values)

print(df_comparacion.head(10))

## 4. Visualización de Predicciones vs Reales

In [ ]:
# Graficar predicciones vs reales para los 3 mejores modelos
modelos_graficar = ['Linear Regression', 'Random Forest', 'Gradient Boosting']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, nombre_modelo in enumerate(modelos_graficar):
    pred = predicciones_test[nombre_modelo]
    
    axes[idx].scatter(y_test, pred, alpha=0.6, s=50)
    axes[idx].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
    axes[idx].set_xlabel('Valor Real')
    axes[idx].set_ylabel('Predicción')
    axes[idx].set_title(f'{nombre_modelo}')
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(project_root, 'results/plots/04_predicciones_vs_reales.png'), dpi=300, bbox_inches='tight')
plt.show()

print("Gráfico guardado")

## 5. Importancia de Features (Modelos basados en árboles)

In [ ]:
# Obtener importancia de features
importance_dict = obtener_features_importance(modelos)

if importance_dict:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    for idx, (nombre_modelo, importances) in enumerate(importance_dict.items()):
        # Crear dataframe con importancias
        df_imp = pd.DataFrame({
            'Feature': feature_names,
            'Importancia': importances
        }).sort_values('Importancia', ascending=True)
        
        # Graficar
        df_imp.plot(kind='barh', x='Feature', y='Importancia', ax=axes[idx], legend=False)
        axes[idx].set_title(f'Importancia de Features - {nombre_modelo}')
        axes[idx].set_xlabel('Importancia')
    
    plt.tight_layout()
    plt.savefig(os.path.join(project_root, 'results/plots/05_feature_importance.png'), dpi=300, bbox_inches='tight')
    plt.show()
    
    print("Gráfico de importancia guardado")
else:
    print("No hay modelos con feature importance")

## 6. Resumen de Modelos

In [ ]:
print("="*70)
print("RESUMEN DE MODELOS SUPERVISADOS ENTRENADOS")
print("="*70)

resumen = []
for nombre, modelo in modelos.items():
    r2_train = modelo.score(X_train, y_train)
    r2_test = modelo.score(X_test, y_test)
    
    resumen.append({
        'Modelo': nombre,
        'R² Train': r2_train,
        'R² Test': r2_test,
        'Diferencia': r2_train - r2_test
    })

df_resumen = pd.DataFrame(resumen).sort_values('R² Test', ascending=False)
print(df_resumen.to_string(index=False))
print("\nMejor modelo en Test:", df_resumen.iloc[0]['Modelo'])

# Guardar resumen
df_resumen.to_csv(os.path.join(project_root, 'results/metrics/02_resumen_modelos.csv'), index=False)
print(f"\nResumen guardado en results/metrics/02_resumen_modelos.csv")